# 4. Benchmark : notre modele vs Google Translate

**Objectif** : comparer notre meilleur modele (LoRA v2) a Google Translate
sur les 241 paires du test de reference verifie.

## Pourquoi ?

- Google Translate couvre l'ewe depuis mai 2022.
- Comparer nos scores a ceux de Google situe notre modele : est-il
  competitif avec un geant commercial ?
- Resultat parlant pour le memoire ("notre modele vs Google Translate").

## Methode

1. Traduire les 241 paires avec Google Translate (les 2 directions)
2. Calculer chrF++ / BLEU (meme metrique que pour nos modeles)
3. Comparer : baseline / v1 / v2 / Google Translate

## Avertissements

- On utilise `deep-translator` (acces NON officiel a l'API web de Google,
  code ewe `ee` gere correctement - contrairement a googletrans qui ne
  supporte pas l'ewe). Si rate limit : alternative = l'API officielle
  Google Cloud (cle gratuite, quota 500k caracteres/mois).
- Les traductions Google ne sont JAMAIS injectees dans notre corpus
  (CGU Google + contamination).
- Pas besoin de GPU : ce notebook tourne sur CPU.

In [ ]:
# Installation
!pip install -q deep-translator sacrebleu pandas

print("Dependances installees")

In [ ]:
# Imports
import time
import pandas as pd
from sacrebleu.metrics import CHRF, BLEU
from deep_translator import GoogleTranslator

print("Pret (execution sur CPU, pas besoin de GPU)")

In [ ]:
# Chargement du test de reference verifie (241 paires)
URL_REF = "https://raw.githubusercontent.com/cherif-tg/tg_nlp_toolkit/main/huggingface/test-reference-final.tsv"
reference = pd.read_csv(URL_REF, sep="\t", on_bad_lines="skip")
print("Reference chargee :", len(reference), "paires")

fr_liste = [str(x) for x in reference["fr"].tolist()]
ew_liste = [str(x) for x in reference["ewe"].tolist()]

In [ ]:
# Traduction via Google Translate (retries + pause anti rate-limit)
def traduire_gt(textes, src, dest, pause=0.3):
    # deep-translator : une instance reutilisable par direction.
    traducteur = GoogleTranslator(source=src, target=dest)
    resultats = []
    for i, t in enumerate(textes):
        ok = False
        for tentative in range(4):
            try:
                resultats.append(traducteur.translate(t))
                ok = True
                break
            except Exception:
                time.sleep(1.5 * (tentative + 1))
        if not ok:
            resultats.append("")
            print(f"  echec ligne {i}: {t[:40]}")
        time.sleep(pause)
        if (i + 1) % 50 == 0:
            print(f"  {i + 1}/{len(textes)}")
    return resultats

print("Fonction de traduction Google prete")

In [ ]:
# Traduction FR -> EWE via Google Translate (code Google : ee)
print("Traduction FR -> EWE (241 paires)...")
gt_fr_ee = traduire_gt(fr_liste, src="fr", dest="ee")
print("Termine. Exemple :")
print("  FR :", fr_liste[0])
print("  GT :", gt_fr_ee[0])
print("  Ref:", ewe_liste[0])

In [ ]:
# Traduction EWE -> FR via Google Translate
print("Traduction EWE -> FR (241 paires)...")
gt_ee_fr = traduire_gt(ewe_liste, src="ee", dest="fr")
print("Termine. Exemple :")
print("  EWE :", ewe_liste[0])
print("  GT  :", gt_ee_fr[0])
print("  Ref :", fr_liste[0])

In [ ]:
# Calcul des scores Google (chrF++ / BLEU)
chrf_metric = CHRF()
bleu_metric = BLEU()

def scorer(preds, refs):
    c = chrf_metric.corpus_score(preds, [refs])
    b = bleu_metric.corpus_score(preds, [refs])
    return round(c.score, 2), round(b.score, 2)

# Filtrer les traductions vides (echecs de traduction)
valides_fr_ee = [(p, r) for p, r in zip(gt_fr_ee, ewe_liste) if p.strip()]
valides_ee_fr = [(p, r) for p, r in zip(gt_ee_fr, fr_liste) if p.strip()]
print(f"FR->EWE : {len(valides_fr_ee)}/{len(gt_fr_ee)} traductions reussies")
print(f"EWE->FR : {len(valides_ee_fr)}/{len(gt_ee_fr)} traductions reussies")

gt_fr_ee_score = scorer([p for p, _ in valides_fr_ee], [r for _, r in valides_fr_ee])
gt_ee_fr_score = scorer([p for p, _ in valides_ee_fr], [r for _, r in valides_ee_fr])
print("Google FR->EWE : chrF++", gt_fr_ee_score[0], "| BLEU", gt_fr_ee_score[1])
print("Google EWE->FR : chrF++", gt_ee_fr_score[0], "| BLEU", gt_ee_fr_score[1])

In [ ]:
# Tableau comparatif complet (baseline / v1 / v2 / Google)
resume = pd.DataFrame({
    "Direction": ["FR->EWE"] * 4 + ["EWE->FR"] * 4,
    "Modele": ["Baseline", "LoRA v1", "LoRA v2", "Google Translate",
               "Baseline", "LoRA v1", "LoRA v2", "Google Translate"],
    "chrF++": [37.22, 47.39, 47.95, gt_fr_ee_score[0],
               38.14, 37.52, 52.24, gt_ee_fr_score[0]],
    "BLEU": [11.17, 22.20, 22.42, gt_fr_ee_score[1],
             14.92, 15.15, 31.83, gt_ee_fr_score[1]],
})
print("=== BENCHMARK : notre modele vs Google Translate (241 paires) ===")
print(resume.to_string(index=False))

# Sauvegarde
resume.to_csv("benchmark-google.csv", index=False, sep=";")
print("Sauvegarde : benchmark-google.csv")

## Lecture des resultats

- Si notre v2 >= Google Translate : notre modele est competitif avec un
  geant commercial sur ce domaine (biblique/generaliste) - argument fort.
- Si Google domine : normal (donnees massives + ewe moderne). Notre valeur
  = open-source, hors-ligne, adapte a l'ewe du Togo, et le kabiyè (que
  Google ne couvre pas).
- Rapporte le tableau pour le memoire et la mise a jour des docs.